# 03e — Feature Engineering (Problem C)

Transform raw BTS and weather data into a model-ready feature matrix.

**Key focus:**
- Temporal cyclical features
- Weather categorization
- **Historical Incident Rates** (Airport, Carrier, Route) — with strict leakage protection.

**Output:** `data/processed/preflight_features_final.parquet`

In [ ]:
import os
from pathlib import Path

root = Path.cwd()
while not (root / "pyproject.toml").exists():
    root = root.parent
os.chdir(root)
print(f"Working directory: {root}")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.features.preflight import add_temporal_features, compute_historical_rates, categorize_weather

IN_PATH = Path('data/processed/preflight_weather_enriched.parquet')
OUT_PATH = Path('data/processed/preflight_features_final.parquet')

df = pd.read_parquet(IN_PATH)
print(f"Loaded dataset with {len(df):,} rows.")

## 1. Temporal and Route Features

In [ ]:
df = add_temporal_features(df)

# Route ID for historical rate tracking
df['route_id'] = df['ORIGIN'] + '_' + df['DEST']

print("Extracted temporal and route IDs.")

## 2. Weather Engineering

In [ ]:
df = categorize_weather(df)

# Fill missing weather with global medians for the POC
for col in ['temp', 'dwpt', 'rhum', 'prcp', 'wspd', 'pres']:
    df[col] = df[col].fillna(df[col].median())

print("Weather features categorized and imputed.")

## 3. Historical Incident Rates (Leakage Protected)

We calculate the incident rate for airports and carriers based on all data available **up to the day before** the current flight.

In [ ]:
print("Computing historical incident rates...")

df['airport_risk_rate'] = compute_historical_rates(df, 'ORIGIN')
df['carrier_risk_rate'] = compute_historical_rates(df, 'OP_UNIQUE_CARRIER')
df['route_risk_rate'] = compute_historical_rates(df, 'route_id')

# Fill initial nulls (where no prior history exists) with 0
df['airport_risk_rate'] = df['airport_risk_rate'].fillna(0)
df['carrier_risk_rate'] = df['carrier_risk_rate'].fillna(0)
df['route_risk_rate'] = df['route_risk_rate'].fillna(0)

print("Historical rates computed.")

## 4. Feature Selection and Correlation

In [ ]:
features = [
    'month_sin', 'month_cos', 'dow_sin', 'dow_cos', 'hour', 
    'temp', 'rhum', 'prcp', 'wspd', 'pres',
    'airport_risk_rate', 'carrier_risk_rate', 'route_risk_rate',
    'DISTANCE'
 ]

plt.figure(figsize=(12, 10))
sns.heatmap(df[features + ['incident']].corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Feature Correlation Matrix")
plt.show()

## 5. Save Final Matrix

In [ ]:
df.to_parquet(OUT_PATH, index=False)
print(f"Saved final feature matrix to {OUT_PATH}")